# Blockchain Audit Layer — Dev Log

## Objetivo e papel no pipeline

`core/blockchain_audit_layer` evolui o `core/audit_logs` do V1 (hash-chain
local) adicionando **checkpoints de Merkle**: agrupa lotes de eventos já
gravados num Merkle root, encadeado ao checkpoint anterior, e permite provas
de inclusão compactas (`O(log n)`) sem reler a cadeia inteira.

**Honestidade de escopo** (ver docstring de `engine.py`): isto reduz, mas
não elimina, a limitação já documentada em `core/audit_logs/CHANGELOG.md` —
um atacante com escrita no arquivo E capacidade de rodar a verificação ainda
pode recalcular tudo do zero. O que este módulo entrega de real é o
pré-requisito técnico para uma ancoragem externa futura: o `merkle_root` de
cada checkpoint é justamente o dado pequeno que se publicaria em algum lugar
fora do controle de quem escreve o log.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import tempfile
from pathlib import Path

from core.audit_logs.logger import AuditLogger
from core.blockchain_audit_layer.engine import create_checkpoint, get_proof, verify_checkpoint_chain
from shared.schemas import AuditEventType

# Isolado em diretório temporário -- nunca toca o log de produção real.
demo_dir = Path(tempfile.mkdtemp(prefix="blockchain_audit_demo_"))
logger = AuditLogger(log_path=demo_dir / "audit_log.jsonl")
checkpoint_path = demo_dir / "checkpoints.jsonl"

for i in range(7):
    logger.record_event(AuditEventType.PII_SCAN, actor="demo", payload={"i": i})

cp1 = create_checkpoint(logger=logger, checkpoint_path=checkpoint_path)
print(f"Checkpoint 1: eventos [{cp1.event_range_start}:{cp1.event_range_end}] -> merkle_root={cp1.merkle_root[:16]}...")

for i in range(3):
    logger.record_event(AuditEventType.POLICY_EVALUATION, actor="demo", payload={"i": i})
cp2 = create_checkpoint(logger=logger, checkpoint_path=checkpoint_path)
print(f"Checkpoint 2: eventos [{cp2.event_range_start}:{cp2.event_range_end}] -> merkle_root={cp2.merkle_root[:16]}...")
print("prev_checkpoint_hash do checkpoint 2 == checkpoint_hash do checkpoint 1?", cp2.prev_checkpoint_hash == cp1.checkpoint_hash)

print("Cadeia de checkpoints íntegra?", verify_checkpoint_chain(checkpoint_path=checkpoint_path))

events = logger.read_events()
proof = get_proof(events[5].hash, logger=logger, checkpoint_path=checkpoint_path)
print(f"Prova de inclusão para o evento índice 5: válida={proof.valid}, {len(proof.proof_path)} passo(s) na prova.")

Checkpoint 1: eventos [0:7] -> merkle_root=60499041e40ed0cb...
Checkpoint 2: eventos [7:10] -> merkle_root=e4cd3b9a8edbb2eb...
prev_checkpoint_hash do checkpoint 2 == checkpoint_hash do checkpoint 1? True
Cadeia de checkpoints íntegra? True
Prova de inclusão para o evento índice 5: válida=True, 3 passo(s) na prova.


O segundo checkpoint cobre só os 3 eventos novos (índices 7-10), e seu
`prev_checkpoint_hash` bate exatamente com o `checkpoint_hash` do primeiro —
a própria cadeia de checkpoints é encadeada, no mesmo padrão do
`audit_logs.AuditLogger`. A prova de inclusão para o evento no meio do
primeiro checkpoint (7 folhas) precisa de só 3 hashes irmãos (`log2(7)`
arredondado), não das 7 folhas inteiras.

## Rodando a suíte de testes

```
"C:/Users/Yuri_/.venvs/athenagov-ai/Scripts/python.exe" -m pytest core/blockchain_audit_layer/tests -v
```

13 testes sobre uma `AuditLogger` real isolada em arquivo temporário: raiz de
Merkle determinística/sensível à ordem, número ímpar de folhas,
range de checkpoints correto, encadeamento entre checkpoints, erro sem
evento novo, detecção de adulteração, provas válidas para todo evento
(inclusive árvore de 1 folha), erro para evento desconhecido, provas
cruzando múltiplos checkpoints.

## Handoff Summary

- **Status:** ✅ done — 13/13 testes passando.
- **TODO onda futura:** ancoragem externa de verdade do `merkle_root`
  (timestamping notarial, git separado, blockchain pública) — hoje só a
  infraestrutura criptográfica local está pronta, a publicação externa não.